# 83일차 · 도구 호출 경로(Trajectory) 평가

승인을 받기 전에 확정 도구를 불러도 마지막 답변은 “처리했습니다”일 수 있습니다. 답변 문구와 별도로 요청 ID, 승인 시점, 실제 저장 건수를 확인합니다.

AgentEvals와 사용자 정의 검사로 도구 선택·인자·선후 관계·중복 호출을 검증한다. 순서를 바꿔도 되는 조회와 승인 후 수행할 확정 작업을 구분하고, 실제 처리 횟수는 앱 기록과 대조한다.

## 도구 인자가 같은 요청을 가리키나요?

In [ ]:
expected_id = "T07"
events = [("lookup_user", "T07"), ("lookup_policy", "T07"), ("approve", "T07"), ("commit", "T08")]
def valid_path(events, request_id, writes):
    approved = False
    commits = 0
    lookups = set()
    for name, target in events:
        if target != request_id:
            return False
        if name not in {"lookup_user", "lookup_policy", "approve", "commit"}:
            return False
        if name.startswith("lookup_"):
            lookups.add(name)
        if name == "approve":
            if lookups != {"lookup_user", "lookup_policy"}:
                return False
            approved = True
        if name == "commit":
            if not approved:
                return False
            commits += 1
    return commits == 1 and writes == [request_id]

print(valid_path(events, expected_id, ["T08"]))

False


승인된 대상은 T07인데 마지막 도구의 인자와 저장 기록은 T08입니다. 승인 이벤트는 앱이 사람의 확인을 기록한 값입니다. 모델이 승인했다고 말한 내용은 여기에 넣지 않습니다.

### 직접 해 보기

독립적인 두 조회 순서만 바꾼 경우, 승인 전에 확정한 경우, 같은 요청을 두 번 저장한 경우를 각각 검사하세요. 마지막 사례에서는 호출 기록은 한 번으로 두세요.

In [ ]:
# valid_path에 실행 사건과 실제 저장 ID 목록을 각각 전달하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>lookups = [(&quot;lookup_user&quot;, &quot;T07&quot;), (&quot;lookup_policy&quot;, &quot;T07&quot;)]
approval, commit = (&quot;approve&quot;, &quot;T07&quot;), (&quot;commit&quot;, &quot;T07&quot;)
print(valid_path(lookups[::-1] + [approval, commit], &quot;T07&quot;, [&quot;T07&quot;]))
print(valid_path(lookups + [commit, approval], &quot;T07&quot;, [&quot;T07&quot;]))
print(valid_path(lookups + [approval, commit], &quot;T07&quot;, [&quot;T07&quot;, &quot;T07&quot;]))</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>True
False
False
</code></pre></details>

## 수업 서비스에 적용하기

agentevals를 설치한 커널에서 아래 셀을 실행하세요. 같은 도구에 다른 요청 ID를 넣었을 때 strict 평가 점수가 어떻게 바뀌는지 확인합니다. 팀 서비스에서는 조회 순서의 교환을 허용할 부분과 승인이 앞서야 할 부분을 나눕니다. LangSmith 메시지에서 읽은 호출과 앱의 저장 로그를 요청 ID로 연결해 두 결과를 제출하세요.

In [ ]:
run_integration = False
if run_integration:
    import json
    from agentevals.trajectory.match import create_trajectory_match_evaluator
    def trajectory(ticket):
        return [{"role": "assistant", "content": "", "tool_calls": [
            {"id":"call1", "type":"function", "function":{
                "name":"lookup", "arguments":json.dumps({"ticket":ticket})}}]}]
    check = create_trajectory_match_evaluator(trajectory_match_mode="strict",
                                              tool_args_match_mode="exact")
    for ticket in ["T07", "T08"]:
        print(ticket, check(outputs=trajectory(ticket), reference_outputs=trajectory("T07"))["score"])

### 제출 전 확인

AgentEvals와 사용자 정의 검사로 도구 선택·인자·선후 관계·중복 호출을 검증한다. 순서를 바꿔도 되는 조회와 승인 후 수행할 확정 작업을 구분하고, 실제 처리 횟수는 앱 기록과 대조한다.

[공식 문서](https://github.com/langchain-ai/agentevals)